# 08 — Funciones avanzadas

## *args y **kwargs

`*args` captura argumentos posicionales extra como tupla. `**kwargs` captura argumentos nombrados extra como diccionario.

In [1]:
def calcular_descuento(precio: float, *descuentos: float, moneda: str = 'EUR') -> str:
    # *descuentos recibe cualquier número de descuentos posicionales
    precio_final = precio
    for d in descuentos:
        precio_final *= (1 - d)
    return f'{precio_final:.2f} {moneda}'

print(calcular_descuento(100, 0.10))              # un descuento
print(calcular_descuento(100, 0.10, 0.05))        # dos descuentos
print(calcular_descuento(100, 0.10, moneda='USD'))# keyword arg explícito


def crear_registro(nombre: str, **atributos) -> dict:
    # **atributos captura cualquier par clave=valor extra
    return {'nombre': nombre, **atributos}

print(crear_registro('Monitor', precio=350, stock=45, color='Negro'))


90.00 EUR
85.50 EUR
90.00 USD
{'nombre': 'Monitor', 'precio': 350, 'stock': 45, 'color': 'Negro'}


## Keyword-only arguments

Parámetros que obligan al caller a usar el nombre explícito — evitan errores de orden.

In [2]:
# Todo lo que va después de * es keyword-only
def exportar_informe(datos, *, formato: str, ruta: str, sobrescribir: bool = False):
    print(f'Exportando {len(datos)} registros → {ruta} ({formato})')
    print(f'Sobrescribir: {sobrescribir}')

# exportar_informe([], 'csv', '/ruta')  # TypeError — formato y ruta son keyword-only
exportar_informe([], formato='csv', ruta='/datos/informe.csv')
exportar_informe([], formato='json', ruta='/api/out.json', sobrescribir=True)


Exportando 0 registros → /datos/informe.csv (csv)
Sobrescribir: False
Exportando 0 registros → /api/out.json (json)
Sobrescribir: True


## Mutable default — el gotcha más común

El valor por defecto se evalúa una sola vez cuando se define la función, no en cada llamada.

In [3]:
# BUG — la lista se comparte entre todas las llamadas
def agregar_venta_bug(monto: float, historial: list = []) -> list:
    historial.append(monto)
    return historial

print(agregar_venta_bug(100))   # [100]
print(agregar_venta_bug(200))   # [100, 200] — la lista se acumuló!

# CORRECTO — usar None como sentinel
def agregar_venta(monto: float, historial: list | None = None) -> list:
    if historial is None:
        historial = []
    historial.append(monto)
    return historial

print(agregar_venta(100))   # [100]
print(agregar_venta(200))   # [200] — lista nueva cada llamada


[100]
[100, 200]
[100]
[200]


## Lambda y funciones de orden superior

Lambda: funciones anónimas de una expresión. `map`, `filter`, `sorted` las aceptan como argumento.

In [4]:
datos = [
    {'nombre': 'Laptop',  'precio': 899, 'unidades': 12},
    {'nombre': 'Mouse',   'precio': 15,  'unidades': 200},
    {'nombre': 'Monitor', 'precio': 350, 'unidades': 45},
]

# sorted con key — lambda extrae el campo de ordenación
por_revenue = sorted(datos, key=lambda x: x['precio'] * x['unidades'], reverse=True)
for p in por_revenue:
    print(f"{p['nombre']:<12} revenue: {p['precio'] * p['unidades']:>8,.0f}")

# map — aplica función a cada elemento, devuelve iterator
precios = [899, 15, 350]
con_iva = list(map(lambda x: round(x * 1.21, 2), precios))
print(con_iva)

# filter — conserva elementos donde la función devuelve True
caros = list(filter(lambda x: x > 100, precios))
print(caros)


Monitor      revenue:   15,750
Laptop       revenue:   10,788
Mouse        revenue:    3,000
[1087.79, 18.15, 423.5]
[899, 350]


## Closures

Una función que captura variables del scope donde fue definida. La variable queda "encerrada" incluso después de que el scope exterior termine.

In [5]:
def crear_calculadora_iva(tasa: float):
    # 'tasa' queda capturada en el closure
    def calcular(precio: float) -> float:
        return round(precio * (1 + tasa), 2)
    return calcular

iva_es = crear_calculadora_iva(0.21)
iva_mx = crear_calculadora_iva(0.16)

print(iva_es(100))   # 121.0
print(iva_mx(100))   # 116.0

# Útil para fábricas de funciones parametrizadas
def multiplicador(factor):
    return lambda x: x * factor

doble  = multiplicador(2)
triple = multiplicador(3)
print(list(map(doble, [1, 2, 3, 4])))   # [2, 4, 6, 8]


121.0
116.0
[2, 4, 6, 8]


## Decoradores

Una función que envuelve otra función para añadir comportamiento sin modificarla. Sintaxis `@nombre`.

In [6]:
import time
from functools import wraps

def medir_tiempo(func):
    @wraps(func)   # preserva __name__ y __doc__ de la función original
    def wrapper(*args, **kwargs):
        inicio  = time.perf_counter()
        resultado = func(*args, **kwargs)
        duracion  = time.perf_counter() - inicio
        print(f'{func.__name__} tardó {duracion:.4f}s')
        return resultado
    return wrapper

@medir_tiempo
def procesar_ventas(n: int) -> int:
    return sum(range(n))

resultado = procesar_ventas(1_000_000)
print(f'Suma: {resultado:,}')


procesar_ventas tardó 0.0174s
Suma: 499,999,500,000


---
## Resumen

| Patrón | Sintaxis |
|--------|----------|
| Posicionales extra | `def f(*args)` |
| Named extra | `def f(**kwargs)` |
| Keyword-only | `def f(a, *, b, c)` |
| Mutable default | `def f(x, lst=None): if lst is None: lst=[]` |
| Lambda | `lambda x: x * 2` |
| Closure | función que captura variables externas |
| Decorador | `@funccion_envoltura` |
